# Topic 12 practice: merge

The tables are built in code, so no CSV is needed. Re-run the setup cell to reset them. Give each result a new name.

Tip: a comment line must start with `#`.

In [ ]:
import pandas as pd

customers = pd.DataFrame({"cust_id": [1, 2, 3, 4, 5],
                          "name":    ["Asha", "Ravi", "Meena", "Dev", "Isha"],
                          "city":    ["Indore", "Pune", "Delhi", "Pune", "Indore"]})

orders = pd.DataFrame({"order_id": [11, 12, 13, 14, 15, 16],
                       "cust_id":  [1, 2, 1, 3, 9, 2],
                       "amount":   [500, 700, 300, 900, 250, 400]})

# a second pair: different key names, a repeated column name, and a duplicate key
sales = pd.DataFrame({"sale_id": [1, 2, 3, 4],
                      "pid":     [10, 20, 10, 30],
                      "city":    ["Indore", "Pune", "Delhi", "Pune"],
                      "qty":     [2, 1, 5, 3]})

products = pd.DataFrame({"product_id": [10, 20, 20, 30],
                         "item":       ["pen", "bag", "bag2", "lamp"],
                         "city":       ["Mumbai", "Mumbai", "Chennai", "Mumbai"],
                         "price":      [20, 450, 460, 300]})
print(customers)
print(orders)
print(sales)
print(products)

### Q1 (easy). Merge and total

Merge `orders` with `customers` on `cust_id` (only matching rows) into `q1`. Print `q1` and its shape. Then print the total `amount` for each customer `name`, and the total `amount` for each `city`.

In [ ]:
q1 = pd.merge(orders, customers, on="cust_id")                 # inner: only keys in both tables
print(q1)
print(q1.shape)                                                # (5, 5): order 15 has no customer, so it is dropped
print(q1.groupby("name")["amount"].sum())                      # Asha 800, Meena 900, Ravi 1100
print(q1.groupby("city")["amount"].sum())                      # Delhi 900, Indore 800, Pune 1100

### Q2 (medium). Keep unmatched rows

Merge `orders` with `customers` keeping ALL orders (`q2`). Print how many orders have no customer, and their `order_id` list. Fill the missing `name` with `unknown` and print `q2`. Then find the names of customers who never ordered anything (print a list). Finally, merge with `indicator=True` and `how="outer"` and print how many rows are `both`, `left_only` and `right_only`.

In [ ]:
q2 = pd.merge(orders, customers, on="cust_id", how="left")    # keep all orders
print(q2["name"].isna().sum())                                 # 1
print(q2[q2["name"].isna()]["order_id"].tolist())               # [15]
q2["name"] = q2["name"].fillna("unknown")                      # fill the gap
print(q2)
nobuy = pd.merge(customers, orders, on="cust_id", how="left")  # start from customers
print(nobuy[nobuy["order_id"].isna()]["name"].tolist())        # ['Dev', 'Isha']
chk = pd.merge(orders, customers, on="cust_id", how="outer", indicator=True)
print(chk["_merge"].value_counts())                            # both 5, right_only 2, left_only 1

### Q3 (medium to hard). Different key names and duplicate keys

`sales` has the key `pid` and `products` has the key `product_id`; both have a column `city`, and `products` has a repeated `product_id` (20 appears twice). First print whether `product_id` is unique in `products`. Merge as it is into `bad` (use the different key names, and suffixes `_sale` and `_prod`) and print `bad` and the total `qty` of `sales` and of `bad`. Then fix it by keeping only the first row of each `product_id`, merge again with `validate="m:1"` into `q3`, drop the extra key column, and print `q3` and its total `qty`.

In [ ]:
print(products["product_id"].is_unique)                          # False -> warning sign
bad = pd.merge(sales, products, left_on="pid", right_on="product_id", suffixes=("_sale", "_prod"))
print(bad)
print(sales["qty"].sum(), bad["qty"].sum())                    # 11 vs 12, the total is wrong
clean = products.drop_duplicates("product_id")                 # keep the first row of each id
q3 = pd.merge(sales, clean, left_on="pid", right_on="product_id",
              suffixes=("_sale", "_prod"), validate="m:1")     # raises an error if duplicates remain
q3 = q3.drop(columns="product_id")                             # drop the extra key column
print(q3)
print(q3["qty"].sum())                                         # 11